# Neural Network Pertama Kamu dengan TensorFlow

Selamat datang di modul Deep Learning! Di notebook ini, kita akan membangun neural network (jaringan saraf tiruan) pertama kita dari nol.

## Tujuan Pembelajaran (Learning Objectives)

Setelah menyelesaikan notebook ini, kamu akan mampu:
- Memahami konsep dasar neural network: neuron, layer, weight, dan bias
- Membangun model Sequential dengan TensorFlow/Keras
- Melatih model dan memantau proses training dengan kurva loss dan accuracy
- Mengevaluasi performa model menggunakan confusion matrix
- Membandingkan arsitektur neural network yang berbeda

**Estimasi waktu**: ~90 menit

**Dataset**: Fashion MNIST — 70,000 gambar pakaian & aksesori


In [ ]:
!pip install -q tensorflow matplotlib seaborn scikit-learn

## Cek GPU (Graphics Processing Unit)

Deep learning membutuhkan **banyak sekali perhitungan matematika** — bayangkan miliaran perkalian dan penjumlahan yang harus dilakukan berulang kali.

**GPU** (Graphics Processing Unit) dirancang khusus untuk komputasi paralel, sehingga bisa memproses ribuan operasi sekaligus. Ini membuat training neural network jauh lebih cepat dibanding menggunakan CPU biasa.

Mari kita cek apakah GPU tersedia di environment kita:


In [ ]:
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
if tf.config.list_physical_devices('GPU'):
    print("\u2705 GPU terdeteksi! Training akan lebih cepat.")
else:
    print("\u26a0\ufe0f Tidak ada GPU. Training tetap bisa jalan, tapi lebih lambat.")

## Bagian 1: Apa itu Neural Network?

### Analogi Otak Manusia

Bayangkan kamu melihat sebuah baju di toko. Secara otomatis matamu menangkap **bentuk**, **warna**, dan **pola** — lalu otak memprosesnya dan kamu langsung tahu: *"Oh ini kaos!"*

**Neural network** (jaringan saraf tiruan) bekerja dengan cara yang mirip:
- **Input** -> menerima data mentah (misalnya: gambar berupa angka)
- **Hidden layers** -> memproses dan mencari pola tersembunyi
- **Output** -> menghasilkan prediksi akhir

### Komponen Utama Neural Network

| Komponen | Penjelasan | Analogi |
|----------|-----------|--------|
| **Neuron** | Unit pemrosesan terkecil | Satu sel saraf di otak |
| **Layer** | Kumpulan neuron | Satu lapisan sel saraf |
| **Weight** (bobot) | Angka yang menentukan kekuatan koneksi | Takaran bumbu dalam resep masakan |
| **Bias** | Angka tambahan untuk menyesuaikan output | Selera dasar koki |

> **Analogi Resep**: Bayangkan neural network seperti koki yang belajar memasak. Weight adalah takaran bumbu (berapa sendok garam, gula, dll), dan bias adalah selera dasar si koki. Proses training adalah koki yang terus bereksperimen hingga masakannya sempurna!


### Arsitektur Model yang Akan Kita Bangun

Berikut adalah struktur neural network yang akan kita buat untuk mengklasifikasi gambar pakaian.
Mari kita visualisasikan arsitekturnya:

In [ ]:
# === Visualisasi Arsitektur Neural Network ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
fig.patch.set_facecolor('#FAFAFA')

# Definisi layer: (y, label, sublabel, color, width_ratio)
layers = [
    (8.5, 'Input', 'Gambar 28 x 28 pixel', '#E3F2FD', 0.8),
    (6.8, 'Flatten', '784 angka dalam satu baris', '#BBDEFB', 1.0),
    (5.1, 'Hidden Layer 1', '128 neuron — aktivasi ReLU', '#42A5F5', 0.85),
    (3.4, 'Hidden Layer 2', '64 neuron — aktivasi ReLU', '#1E88E5', 0.65),
    (1.7, 'Output Layer', '10 kategori — aktivasi Softmax', '#1565C0', 0.5),
]

box_h = 1.0
center_x = 5.0

for y, label, sublabel, color, w_ratio in layers:
    w = 6.0 * w_ratio
    x = center_x - w / 2
    text_color = 'white' if color.startswith('#1') or color.startswith('#4') else '#333'
    rect = mpatches.FancyBboxPatch(
        (x, y), w, box_h,
        boxstyle='round,pad=0.15',
        facecolor=color, edgecolor='#0D47A1', linewidth=1.5
    )
    ax.add_patch(rect)
    ax.text(center_x, y + box_h * 0.62, label,
            ha='center', va='center', fontsize=13, fontweight='bold', color=text_color)
    ax.text(center_x, y + box_h * 0.3, sublabel,
            ha='center', va='center', fontsize=10, color=text_color, style='italic')

# Panah antar layer
arrow_style = dict(arrowstyle='->', color='#0D47A1', lw=2.5)
for i in range(len(layers) - 1):
    y_from = layers[i][0]
    y_to   = layers[i + 1][0] + box_h
    ax.annotate('', xy=(center_x, y_to), xytext=(center_x, y_from),
                arrowprops=arrow_style)

# Label prediksi di bawah
ax.text(center_x, 0.7, '\u2193  Prediksi: "Ini Kemeja!"',
        ha='center', va='center', fontsize=13, fontweight='bold', color='#0D47A1')

# Judul
ax.set_title('Arsitektur Neural Network untuk Klasifikasi Fashion',
             fontsize=14, fontweight='bold', pad=15)

# Keterangan neuron di samping kanan
notes = [
    (8.5, '28 x 28 = 784 pixel'),
    (6.8, '784 input features'),
    (5.1, '100,480 parameter'),
    (3.4, '8,256 parameter'),
    (1.7, '650 parameter'),
]
for y, note in notes:
    ax.text(9.2, y + box_h / 2, note,
            ha='center', va='center', fontsize=8, color='#666',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF9C4', edgecolor='#FBC02D', alpha=0.8))

plt.tight_layout()
plt.show()

print('\nSetiap layer memproses informasi dari layer sebelumnya')
print('dan meneruskan hasilnya ke layer berikutnya.')
print('Seperti rantai pengolahan informasi! \U0001f517')

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("✅ Library berhasil diimport!")

## Bagian 2: Dataset Fashion MNIST

### Tentang Dataset

**Fashion MNIST** adalah dataset yang berisi **70,000 gambar** pakaian dan aksesori dalam format:
- Ukuran: **28x28 pixel** (grayscale/hitam-putih)
- Terdiri dari **10 kategori** berbeda
- Dibagi menjadi: 60,000 data training + 10,000 data testing

Dataset ini sangat populer untuk belajar deep learning karena:
- Cukup sederhana untuk dipahami pemula
- Lebih menantang dari MNIST angka (digit 0-9)
- Ukurannya tidak terlalu besar, sehingga cepat diproses

**10 Kategori Pakaian:**

| No | Kategori | No | Kategori |
|----|---------|-----|----------|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Celana (Trouser) | 6 | Kemeja (Shirt) |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Tas (Bag) |
| 4 | Coat | 9 | Ankle boot |


In [ ]:
# Load dataset Fashion MNIST
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Pisahkan 5000 data terakhir dari training sebagai validasi
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

# Nama-nama kategori dalam bahasa Indonesia
class_names = ["T-shirt/top", "Celana", "Pullover", "Dress", "Coat",
               "Sandal", "Kemeja", "Sneaker", "Tas", "Ankle boot"]

print("\U0001f4e6 Dataset berhasil dimuat!")
print(f"Data training  : {X_train.shape[0]:,} gambar")
print(f"Data validasi  : {X_valid.shape[0]:,} gambar")
print(f"Data testing   : {X_test.shape[0]:,} gambar")
print(f"Ukuran gambar  : {X_train.shape[1]}x{X_train.shape[2]} pixel")
print(f"Jumlah kelas   : {len(class_names)}")
print(f"Tipe data      : {X_train.dtype}")
print(f"Nilai pixel    : {X_train.min()} - {X_train.max()}")

Mari kita lihat contoh gambar dari dataset ini untuk memahami apa yang akan dipelajari model:


In [ ]:
# Tampilkan 20 contoh gambar dari dataset
plt.figure(figsize=(12, 10))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray_r')
    plt.title(class_names[y_train[i]], fontsize=11)
    plt.axis('off')
plt.suptitle("Contoh Gambar dari Fashion MNIST", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Bagian 3: Normalisasi Data (Data Normalization)

### Mengapa Perlu Normalisasi?

Bayangkan kamu ingin membandingkan tinggi badan (misalnya 165 cm) dengan berat badan (misalnya 55 kg). Angkanya berbeda jauh dalam skala!

Neural network bisa "bingung" saat menerima angka dengan skala yang sangat berbeda. Nilai pixel saat ini berkisar antara **0 sampai 255**. Kita perlu mengubahnya menjadi rentang **0.0 sampai 1.0** dengan cara membagi 255.

**Manfaat normalisasi:**
- Training lebih stabil dan cepat
- Model lebih mudah menemukan pola
- Menghindari masalah angka yang terlalu besar (exploding gradient)


In [ ]:
# Tampilkan nilai sebelum normalisasi
print("Sebelum normalisasi:")
print(f"  Nilai minimum : {X_train.min()}")
print(f"  Nilai maksimum: {X_train.max()}")
print(f"  Tipe data     : {X_train.dtype}")

# Normalisasi: bagi dengan 255 agar nilai menjadi 0.0 - 1.0
X_train = X_train / 255.0
X_valid = X_valid / 255.0
X_test  = X_test  / 255.0

print("\nSesudah normalisasi:")
print(f"  Nilai minimum : {X_train.min():.1f}")
print(f"  Nilai maksimum: {X_train.max():.1f}")
print(f"  Tipe data     : {X_train.dtype}")

Sekarang semua nilai pixel berada di rentang **0.0 hingga 1.0**. Neural network akan jauh lebih mudah belajar dari data yang sudah dinormalisasi!

> **Tips**: Normalisasi adalah langkah preprocessing yang hampir selalu dilakukan dalam deep learning. Selain pembagian dengan nilai maksimum, ada juga teknik lain seperti standardisasi (mean=0, std=1).


## Bagian 4: Membangun Model (Building the Model)

### Penjelasan Setiap Layer

**1. Flatten Layer**
- Gambar berukuran 28x28 = **784 pixel**
- Flatten "membuka" gambar 2D menjadi **satu baris panjang** berisi 784 angka
- Seperti menggelar karpet yang terlipat menjadi lurus

**2. Dense(128, activation='relu') — Hidden Layer 1**
- **128 neuron** yang masing-masing terhubung ke semua 784 input
- **ReLU** (Rectified Linear Unit) = fungsi aktivasi: jika nilai negatif -> jadikan 0, jika positif -> biarkan
- Seperti filter yang hanya meneruskan sinyal yang "bermakna"
- Rumus ReLU: `f(x) = max(0, x)`

**3. Dense(64, activation='relu') — Hidden Layer 2**
- Layer kedua lebih kecil: **64 neuron**
- Seperti "menyaring" informasi terpenting dari layer sebelumnya
- Membantu model memahami fitur yang lebih abstrak

**4. Dense(10, activation='softmax') — Output Layer**
- **10 output** = satu untuk setiap kategori pakaian
- **Softmax** mengubah angka menjadi **probabilitas** (total semua = 100%)
- Contoh output: [0.02, 0.01, 0.05, 0.01, 0.01, 0.02, **0.85**, 0.01, 0.01, 0.01] -> "Kemeja 85%"


In [ ]:
# Bangun model Sequential
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu', name='hidden_1'),
    tf.keras.layers.Dense(64,  activation='relu', name='hidden_2'),
    tf.keras.layers.Dense(10,  activation='softmax', name='output')
])

model.summary()

### Visualisasi Model dengan `plot_model()`

TensorFlow/Keras punya fitur bawaan untuk menggambar arsitektur model secara otomatis.
Ini sangat berguna untuk model yang lebih kompleks nanti.

In [ ]:
# Visualisasi otomatis dari Keras
tf.keras.utils.plot_model(
    model,
    show_shapes=True,
    show_layer_names=True,
    show_layer_activations=True,
    to_file='model_architecture.png',
    dpi=100
)
# plot_model() juga menyimpan gambar ke file 'model_architecture.png'

### Memahami Jumlah Parameter

Perhatikan kolom **Param #** pada output di atas:

- **Hidden Layer 1**: 784 x 128 + 128 (bias) = **100,480 parameter**
- **Hidden Layer 2**: 128 x 64 + 64 (bias) = **8,256 parameter**
- **Output Layer**: 64 x 10 + 10 (bias) = **650 parameter**

Setiap parameter adalah satu **nilai yang dipelajari** dari data. Proses training = mencari nilai optimal untuk semua parameter ini!


In [ ]:
# Hitung dan tampilkan informasi parameter
total_params = model.count_params()
print(f"\n\U0001f4ca Informasi Parameter Model:")
print(f"   Total parameter yang harus dipelajari: {total_params:,}")
print(f"\n\U0001f4a1 Setiap parameter adalah satu 'keputusan kecil' yang model pelajari dari data.")
print(f"   Model ini perlu mengoptimalkan {total_params:,} angka sekaligus!")
print(f"\nPerbandingan ukuran model:")
print(f"   Model kita           : {total_params:,} parameter")
print(f"   GPT-2 (2019)         : 117,000,000 parameter")
print(f"   GPT-3 (2020)         : 175,000,000,000 parameter")

## Bagian 5: Compile dan Training Model

### Sebelum Training: Tentukan Strategi Belajar

Sebelum model mulai belajar, kita perlu menentukan tiga hal penting:

**1. Loss Function (Fungsi Kerugian)** — `sparse_categorical_crossentropy`
- Mengukur **seberapa salah** prediksi model
- Semakin kecil nilainya, semakin bagus prediksi model
- "Sparse" karena label kita berupa angka (0-9), bukan one-hot encoding

**2. Optimizer** — `adam`
- Strategi untuk **memperbaiki kesalahan** dan memperbarui parameter
- **Adam** (Adaptive Moment Estimation) adalah optimizer yang pintar — dia menyesuaikan kecepatan belajar secara otomatis
- Jauh lebih efektif dari SGD (Stochastic Gradient Descent) biasa untuk pemula

**3. Metrics** — `accuracy`
- Cara mengukur **kemajuan** model: berapa persen prediksi yang benar?


In [ ]:
# Compile model — tentukan cara belajar
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
print("\u2705 Model siap dilatih!")
print("   Loss function : sparse_categorical_crossentropy")
print("   Optimizer     : Adam")
print("   Metrics       : accuracy")

### Waktunya Training!

Proses training adalah inti dari machine learning. Model akan:
1. Melihat batch gambar (32 gambar sekaligus)
2. Membuat prediksi
3. Menghitung kesalahan (loss)
4. Memperbaiki parameter menggunakan optimizer
5. Mengulang proses ini untuk semua data -> ini disebut **1 epoch**

Kita akan melatih selama **20 epoch** — artinya model akan melihat seluruh data training sebanyak 20 kali!


In [ ]:
print("\U0001f3cb\ufe0f Mulai training...")
print(f"   Data training  : {X_train.shape[0]:,} gambar")
print(f"   Epochs         : 20")
print(f"   Batch size     : 32 gambar per iterasi")
print(f"   Iterasi/epoch  : {X_train.shape[0]//32:,}")
print()

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_valid, y_valid),
    verbose=1
)
print("\n\u2705 Training selesai!")

## Bagian 6: Visualisasi Proses Training

Bagaimana model belajar dari waktu ke waktu? Mari kita plot **kurva training** untuk melihat perkembangan model.

Yang kita harapkan:
- **Loss** terus menurun — model semakin berkurang kesalahannya
- **Accuracy** terus meningkat — model semakin sering benar
- **Training** dan **Validation** tidak terlalu jauh berbeda — model tidak overfitting


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Kurva Loss
ax1.plot(history.history['loss'], label='Training Loss', linewidth=2, color='#E53935')
ax1.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='#FB8C00', linestyle='--')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss (Kesalahan)', fontsize=12)
ax1.set_title('Kurva Loss (Kesalahan)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Kurva Accuracy
ax2.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2, color='#1E88E5')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='#43A047', linestyle='--')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (Ketepatan)', fontsize=12)
ax2.set_title('Kurva Accuracy (Ketepatan)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle('Proses Belajar Neural Network selama 20 Epoch', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Ringkasan akhir training
final_train_acc = history.history['accuracy'][-1]
final_val_acc   = history.history['val_accuracy'][-1]
print(f"\n\U0001f4ca Hasil Training:")
print(f"   Akurasi Training  : {final_train_acc:.1%}")
print(f"   Akurasi Validasi  : {final_val_acc:.1%}")

## Bagian 7: Evaluasi Model (Model Evaluation)

### Ujian Akhir dengan Data Test

Sekarang kita uji model dengan **data yang BELUM PERNAH dilihat** selama training (test set).

Ini seperti **ujian akhir** di sekolah — apakah siswa benar-benar memahami materi, atau hanya menghafal soal latihan?

Kita juga akan membuat **Confusion Matrix** untuk melihat:
- Kategori mana yang sering diprediksi dengan benar
- Kategori mana yang sering tertukar satu sama lain


In [ ]:
# Evaluasi pada data test
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\U0001f4ca Hasil Evaluasi pada Data Test (yang belum pernah dilihat):")
print(f"   Akurasi : {test_acc:.1%}")
print(f"   Loss    : {test_loss:.4f}")

# Buat prediksi untuk semua data test
y_pred = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5)
plt.xlabel('Prediksi Model', fontsize=12)
plt.ylabel('Label Sebenarnya', fontsize=12)
plt.title('Confusion Matrix — Mana yang Sering Salah?', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

### Cara Membaca Confusion Matrix

- **Diagonal** (kiri atas ke kanan bawah) = prediksi **BENAR** — semakin gelap warnanya, semakin banyak yang benar
- **Di luar diagonal** = prediksi **SALAH** — ini yang perlu kita perhatikan

**Pengamatan umum pada Fashion MNIST:**
- "Kemeja" (Shirt) dan "T-shirt/top" sering tertukar karena secara visual memang mirip
- "Pullover" dan "Coat" juga kadang tertukar
- "Tas" (Bag), "Celana" (Trouser), dan "Ankle boot" biasanya diprediksi dengan sangat akurat karena bentuknya unik

> **Insight**: Bahkan manusia pun kadang sulit membedakan kemeja dan t-shirt dari gambar kecil 28x28 pixel!


## Bagian 8: Prediksi Individual (Individual Prediction)

Mari kita coba prediksi beberapa gambar satu per satu dan lihat seberapa **percaya diri** (confidence) model dalam membuat keputusan.

Bar chart di bawah setiap gambar menunjukkan probabilitas untuk setiap kategori — semakin panjang bar-nya, semakin yakin model.


In [ ]:
def prediksi_gambar(model, images, labels, class_names, n=8):
    """Tampilkan prediksi model beserta confidence untuk beberapa gambar"""
    predictions = model.predict(images[:n], verbose=0)
    
    fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 6))
    
    for i in range(n):
        # Tampilkan gambar
        axes[0, i].imshow(images[i], cmap='gray_r')
        pred_class = np.argmax(predictions[i])
        true_class = labels[i]
        color = 'green' if pred_class == true_class else 'red'
        axes[0, i].set_title(
            f"{class_names[pred_class]}",
            color=color, fontsize=9, fontweight='bold'
        )
        axes[0, i].axis('off')
        
        # Bar chart confidence
        bar_colors = ['steelblue'] * 10
        bar_colors[pred_class] = color
        axes[1, i].barh(range(10), predictions[i], color=bar_colors)
        axes[1, i].set_xlim(0, 1)
        axes[1, i].set_yticks(range(10))
        axes[1, i].set_yticklabels(class_names, fontsize=7)
        axes[1, i].tick_params(axis='x', labelsize=7)
        conf = predictions[i][pred_class]
        axes[1, i].set_xlabel(f'{conf:.0%}', fontsize=8, color=color, fontweight='bold')
    
    plt.suptitle(
        'Prediksi Model  |  Hijau = Benar  |  Merah = Salah',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

prediksi_gambar(model, X_test, y_test, class_names)

Neural network memberikan **probabilitas untuk SETIAP kelas** secara bersamaan. Kelas dengan probabilitas tertinggi menjadi prediksi akhir.

Perhatikan bahwa ketika model "ragu-ragu", bar chart akan menunjukkan beberapa kelas dengan probabilitas yang hampir sama — ini tanda model tidak terlalu yakin dengan prediksinya.


## Bagian 9: Eksperimen — Perbandingan Arsitektur

### Apa yang Terjadi Kalau Kita Ubah Arsitektur?

Pertanyaan yang sering muncul: **Apakah model lebih besar selalu lebih bagus?**

Mari kita bereksperimen dengan tiga ukuran arsitektur berbeda dan bandingkan hasilnya:

| Model | Arsitektur | Parameter |
|-------|-----------|----------|
| Kecil | 1 hidden layer: [64] | ~51K |
| Sedang | 2 hidden layers: [128, 64] | ~109K |
| Besar | 3 hidden layers: [256, 128, 64] | ~235K |


In [ ]:
def buat_dan_latih_model(nama, layers_config, epochs=15):
    """Buat model dengan konfigurasi tertentu, latih, dan kembalikan hasil"""
    # Bangun arsitektur
    layers = [tf.keras.layers.Flatten(input_shape=(28, 28))]
    for units in layers_config:
        layers.append(tf.keras.layers.Dense(units, activation='relu'))
    layers.append(tf.keras.layers.Dense(10, activation='softmax'))
    
    m = tf.keras.Sequential(layers)
    m.compile(
        loss='sparse_categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    
    print(f"\n{'='*55}")
    print(f"\U0001f3d7\ufe0f  Model: {nama}")
    print(f"   Arsitektur   : {layers_config}")
    print(f"   Total param  : {m.count_params():,}")
    print(f"{'='*55}")
    
    h = m.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=32,
        validation_data=(X_valid, y_valid),
        verbose=0
    )
    
    _, test_acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"   Akurasi test : {test_acc:.1%}")
    return h, test_acc

# Jalankan eksperimen tiga arsitektur
h1, acc1 = buat_dan_latih_model("Kecil  (1 hidden layer)",  [64],        epochs=15)
h2, acc2 = buat_dan_latih_model("Sedang (2 hidden layers)", [128, 64],   epochs=15)
h3, acc3 = buat_dan_latih_model("Besar  (3 hidden layers)", [256, 128, 64], epochs=15)

**Insight Penting:**

Model yang lebih besar tidak selalu menghasilkan akurasi yang jauh lebih tinggi, terutama untuk dataset sederhana seperti Fashion MNIST. Menambah layer/neuron juga berarti:
- Waktu training lebih lama
- Risiko **overfitting** lebih tinggi (model menghafal data training, bukan memahami pola)
- Butuh lebih banyak data untuk melatih secara efektif

Prinsip **Occam's Razor** berlaku di sini: gunakan model sesederhana mungkin yang masih memberikan hasil cukup baik.


In [ ]:
# Visualisasi perbandingan akurasi ketiga arsitektur
models_name = ['Kecil\n(1 layer)', 'Sedang\n(2 layer)', 'Besar\n(3 layer)']
accuracies  = [acc1, acc2, acc3]
colors      = ['#42A5F5', '#66BB6A', '#FFA726']

plt.figure(figsize=(8, 5))
bars = plt.bar(models_name, accuracies, color=colors, edgecolor='white', linewidth=2, width=0.5)

for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f'{acc:.1%}',
        ha='center', va='bottom', fontweight='bold', fontsize=13
    )

plt.ylim(0.80, 0.93)
plt.ylabel('Akurasi Test', fontsize=12)
plt.title('Perbandingan Akurasi Tiga Arsitektur Neural Network',
          fontsize=13, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n\U0001f4cc Kesimpulan Eksperimen:")
best_idx = np.argmax(accuracies)
best_name = models_name[best_idx].replace('\n', ' ')
print(f"   Model terbaik : {best_name}")
print(f"   Selisih kecil vs besar: {abs(acc3 - acc1):.1%}")
print(f"   Model sederhana sudah memberikan hasil yang kompetitif!")

## Bagian 10: Peran NVIDIA dalam Deep Learning

### Mengapa GPU NVIDIA Sangat Penting?

Neural network yang kita latih tadi memiliki sekitar **100,000 parameter**. Model modern seperti GPT-4 memiliki **TRILIUNAN parameter!** Tanpa GPU yang powerful, training model seperti itu akan memakan waktu bertahun-tahun.

**Kenapa GPU lebih cepat dari CPU untuk deep learning?**
- **CPU** punya 4-32 core yang sangat kuat untuk tugas kompleks
- **GPU NVIDIA** punya **ribuan core** yang lebih kecil, tapi sangat efisien untuk perhitungan paralel
- Perkalian matriks (operasi utama neural network) sangat cocok diparalelkan di GPU

**Teknologi NVIDIA untuk AI:**
- **CUDA** — platform komputasi paralel yang digunakan TensorFlow dan PyTorch
- **cuDNN** — library deep learning yang dioptimalkan untuk GPU NVIDIA
- **Tensor Cores** — unit hardware khusus untuk operasi matriks di GPU modern (A100, H100)
- **Google Colab** menggunakan GPU NVIDIA **Tesla T4** — GPU yang sama digunakan di data center!

> Framework TensorFlow yang kita gunakan sudah terintegrasi otomatis dengan CUDA. Ketika GPU tersedia, TensorFlow akan menggunakannya tanpa perlu konfigurasi tambahan.


In [ ]:
import time

if tf.config.list_physical_devices('GPU'):
    gpu_device = tf.config.list_physical_devices('GPU')[0]
    print(f"\U0001f532 GPU yang digunakan: {gpu_device}")
    
    size = 5000
    a = tf.random.normal([size, size])
    b = tf.random.normal([size, size])
    
    # Warmup GPU
    with tf.device('/GPU:0'):
        _ = tf.matmul(a, b).numpy()
    
    # Benchmark GPU
    start = time.time()
    with tf.device('/GPU:0'):
        c = tf.matmul(a, b)
        _ = c.numpy()
    gpu_time = time.time() - start
    
    # Benchmark CPU
    start = time.time()
    with tf.device('/CPU:0'):
        c = tf.matmul(a, b)
        _ = c.numpy()
    cpu_time = time.time() - start
    
    speedup = cpu_time / gpu_time
    
    print(f"\n\u23f1\ufe0f  Benchmark Perkalian Matriks {size}x{size}:")
    print(f"   CPU : {cpu_time:.3f} detik")
    print(f"   GPU : {gpu_time:.3f} detik")
    print(f"   GPU {speedup:.1f}x lebih cepat dari CPU! \U0001f680")
    print(f"\n\U0001f4a1 Bayangkan jika training model GPT-3 (175 miliar parameter) tanpa GPU...")
    print(f"   Itu akan membutuhkan waktu ratusan tahun di CPU biasa!")
else:
    print("\U0001f4a1 GPU tidak terdeteksi di environment ini.")
    print("   Untuk mengaktifkan GPU di Google Colab:")
    print("   Runtime -> Change runtime type -> Hardware accelerator -> GPU")
    print("\n   Dengan GPU NVIDIA Tesla T4 (tersedia gratis di Colab):")
    print("   Training neural network bisa 5-10x lebih cepat!")

## Kesimpulan (Summary)

Selamat! Kamu telah berhasil menyelesaikan notebook ini. Mari kita rangkum apa yang sudah dipelajari:

### Yang Sudah Kita Pelajari

- Membangun neural network pertama dengan TensorFlow
- Neural network bekerja mirip otak manusia: menerima input, memproses di hidden layers, menghasilkan output prediksi
- Arsitektur Sequential: Flatten -> Dense(128, ReLU) -> Dense(64, ReLU) -> Dense(10, Softmax)
- Normalisasi data penting agar training lebih stabil (0-255 menjadi 0.0-1.0)
- Compile model membutuhkan tiga komponen: loss function, optimizer, dan metrics
- Training adalah proses model memperbaiki parameter berulang kali selama beberapa epoch
- Confusion Matrix membantu kita memahami kesalahan spesifik yang dibuat model
- Arsitektur lebih besar tidak selalu berarti akurasi lebih tinggi
- GPU NVIDIA mempercepat training neural network secara drastis

### Konsep Kunci

| Istilah | Arti |
|---------|------|
| Epoch | Satu putaran lengkap melihat semua data training |
| Batch size | Jumlah sampel yang diproses sekaligus |
| ReLU | Fungsi aktivasi: `max(0, x)` |
| Softmax | Mengubah angka menjadi probabilitas (total = 100%) |
| Overfitting | Model terlalu "hafal" data training, buruk di data baru |

### Selanjutnya

Di notebook berikutnya, kita akan mendalami **Activation Functions** — berbagai fungsi aktivasi yang membuat neural network mampu mempelajari pola yang sangat kompleks. Kita akan membandingkan ReLU, Sigmoid, Tanh, dan variannya!

> **Tantangan Opsional**: Coba ubah arsitektur model utama — tambahkan layer baru atau ubah jumlah neuron. Apakah akurasi meningkat? Apakah ada tanda-tanda overfitting?
